# 31. 첫 반응군 층화 샘플링

**목적:** 리뷰 텍스트 분석을 위해 첫 반응군(Steam 총 리뷰 10~49개) 게임에서 대표 표본을 추출한다.

**방법:** `01_stratified_sampling.ipynb`와 동일한 로직 적용
- 대표 장르: 희귀 장르 우선 선정
- 층화 기준: 장르(8개) × 리뷰 신뢰도(2단계: low/mid)
- 신뢰도 high(381개+)는 첫 반응군(최대 49개) 내에 존재하지 않음

**출력:** `data/preprocessed/steam_indie_first_response_sample.csv`

In [ ]:
import ast
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

DATA_DIR    = '../../../data/preprocessed'
TOTAL_N     = 200
MIN_FLOOR   = 5
RANDOM_SEED = 42
TARGET_GENRES = {
    'Action', 'Adventure', 'Casual',
    'Simulation', 'RPG', 'Strategy', 'Sports', 'Racing',
}

print('설정 완료')

## 1. 데이터 로드 및 첫 반응군 필터링

In [ ]:
df_all = pd.read_csv(f'{DATA_DIR}/steam_indie_games.csv')

def parse_genres(v):
    try:
        return ast.literal_eval(v)
    except Exception:
        return []

df_all['genres'] = df_all['genres'].apply(parse_genres)
df_all['positive_rate'] = df_all['positive'] / df_all['total_reviews'] * 100

# 첫 반응군 필터링
df = df_all[(df_all['total_reviews'] >= 10) & (df_all['total_reviews'] < 50)].copy()

print(f'전체 반응군   : {len(df_all):,}개')
print(f'첫 반응군(10~49개): {len(df):,}개')
print()
print('total_reviews 분포:')
print(df['total_reviews'].describe().round(1))

## 2. 대표 장르 선정 (희귀 장르 우선)

In [ ]:
# Indie 제외 후 대상 장르만 보유한 게임 필터
df['genres_filtered'] = df['genres'].apply(
    lambda g: [x for x in g if x != 'Indie' and x in TARGET_GENRES]
)
df = df[df['genres_filtered'].map(len) > 0].copy()

# 희귀도 계산
genre_count = Counter(
    g for genres in df['genres_filtered'] for g in genres
)
print('장르별 게임 수 (희귀도 기준 오름차순):')
for genre, cnt in sorted(genre_count.items(), key=lambda x: x[1]):
    print(f'  {genre:<15}: {cnt:,}')

# 희귀 장르 우선 대표 장르 배정
df['primary_genre'] = df['genres_filtered'].apply(
    lambda genres: min(genres, key=lambda g: genre_count[g])
)

print('\n대표 장르 분포:')
print(df['primary_genre'].value_counts().to_string())

## 3. 리뷰 신뢰도 층 할당

Wilson Score 95% CI 최대 오차 기준 경계값 (기존과 동일):
- `low` : total_reviews < 39 (오차 ±15% 초과)
- `mid` : 39 ≤ total_reviews < 381 (오차 ±5~15%)
- `high`: total_reviews ≥ 381 → 첫 반응군(최대 49개) 내에 없음

In [ ]:
Z = 1.96

def wilson_margin(n):
    p = 0.5
    denom = 1 + Z**2 / n
    return (Z / denom) * np.sqrt(p * (1-p) / n + Z**2 / (4 * n**2)) * 100

def find_n_for_margin(target_pct):
    for n in range(1, 10000):
        if wilson_margin(n) <= target_pct:
            return n

LOW_BOUNDARY  = find_n_for_margin(15)
HIGH_BOUNDARY = find_n_for_margin(5)

print(f'low  (±15% 초과): total_reviews < {LOW_BOUNDARY}')
print(f'mid  (±5~15%)   : {LOW_BOUNDARY} ≤ total_reviews < {HIGH_BOUNDARY}')
print(f'high (±5% 이하) : total_reviews ≥ {HIGH_BOUNDARY}  ← 첫 반응군 내 없음')

def assign_trust(n):
    if n >= HIGH_BOUNDARY:
        return 'high'
    elif n >= LOW_BOUNDARY:
        return 'mid'
    else:
        return 'low'

df['trust']   = df['total_reviews'].apply(assign_trust)
df['stratum'] = df['primary_genre'] + '_' + df['trust']

print('\ntrust 분포:')
print(df['trust'].value_counts())

## 4. 층별 게임 수 확인

In [ ]:
pop = df['stratum'].value_counts().sort_index()
N   = len(df)

print(f'모집단: {N:,}개  |  층 수: {len(pop)}개\n')
print(f'{"층":<22} {"게임 수":>8}  {"비중":>7}')
print('-' * 42)
for stratum, cnt in pop.items():
    print(f'{stratum:<22} {cnt:>8,}  {cnt/N*100:>6.1f}%')
print('-' * 42)
print(f'{"합계":<22} {N:>8,}  {100.0:>6.1f}%')

## 5. 표본 배분 — 비례 배분 + 최소 하한선

In [ ]:
def compute_sample_plan(pop_series, total_n, min_floor):
    proportional = (pop_series / pop_series.sum() * total_n).round().astype(int)
    floored = proportional.clip(lower=min_floor)
    floored = floored.combine(pop_series, min)  # 모집단 수 초과 방지
    overflow = floored.sum() - total_n
    if overflow > 0:
        reducible = floored[(floored > min_floor) & (floored < pop_series)]
        if len(reducible) > 0:
            above = reducible - min_floor
            cut   = (above / above.sum() * overflow).round().astype(int)
            diff  = cut.sum() - overflow
            if diff != 0:
                cut.iloc[cut.argmax()] -= diff
            floored[reducible.index] -= cut
    return floored

sample_plan  = compute_sample_plan(pop, TOTAL_N, MIN_FLOOR)
proportional = (pop / pop.sum() * TOTAL_N).round().astype(int)

print(f'목표 표본: {TOTAL_N}개  |  층당 최소: {MIN_FLOOR}개\n')
print(f'{"층":<22} {"모집단":>8}  {"비중":>7}  {"비례":>6}  {"최종":>6}  {"추출률":>7}  비고')
print('-' * 76)
for stratum in pop.index:
    cnt  = pop[stratum]
    prop = proportional[stratum]
    n    = sample_plan[stratum]
    rate = n / cnt * 100
    note = '전수' if n == cnt else ('하한' if n == MIN_FLOOR and prop < MIN_FLOOR else '')
    print(f'{stratum:<22} {cnt:>8,}  {cnt/N*100:>6.1f}%  {prop:>6}  {n:>6}  {rate:>6.1f}%  {note}')
print('-' * 76)
print(f'{"합계":<22} {N:>8,}  {"100.0%":>7}  {proportional.sum():>6}  {sample_plan.sum():>6}')

## 6. 층화 추출

In [ ]:
sampled_frames = []
for stratum, n in sample_plan.items():
    pool     = df[df['stratum'] == stratum]
    actual_n = min(n, len(pool))
    sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
    sampled_frames.append(sample)

df_sample = pd.concat(sampled_frames).reset_index(drop=True)
print(f'추출 완료: {len(df_sample)}개')
print(df_sample['stratum'].value_counts().sort_index())

## 7. 검증

In [ ]:
pop_ratio  = df['stratum'].value_counts(normalize=True).sort_index() * 100
samp_ratio = df_sample['stratum'].value_counts(normalize=True).sort_index() * 100
samp_count = df_sample['stratum'].value_counts().sort_index()

print(f'{"층":<22} {"모집단%":>8}  {"표본%":>7}  {"격차":>7}  {"표본n":>6}')
print('-' * 58)
for stratum in pop_ratio.index:
    p   = pop_ratio[stratum]
    s   = samp_ratio.get(stratum, 0)
    n   = samp_count.get(stratum, 0)
    gap = s - p
    flag = ' ⚠' if abs(gap) > 10 else ''
    print(f'{stratum:<22} {p:>7.1f}%  {s:>6.1f}%  {gap:>+6.1f}%  {n:>6}{flag}')
print('-' * 58)

checks = [
    ('층당 최소 하한 충족', all(samp_count >= MIN_FLOOR)),
    ('총 표본 수 일치',    len(df_sample) == TOTAL_N),
]
print()
for label, ok in checks:
    print(f'  [{"✓" if ok else "✗"}] {label}')

## 8. 저장

In [ ]:
OUT_COLS = [
    'appid', 'name', 'release_date', 'genres',
    'positive', 'negative', 'total_reviews', 'positive_rate', 'price',
    'developers', 'primary_genre', 'trust', 'stratum',
]

out_path = f'{DATA_DIR}/steam_indie_first_response_sample.csv'
df_sample[OUT_COLS].to_csv(out_path, index=False)
print(f'저장 완료 → {out_path}')
print(f'행 수: {len(df_sample)}개  |  컬럼: {len(OUT_COLS)}개')
df_sample[OUT_COLS].head(3)